# DiFuMo MLP/GAT Saved-Artifact Comparison

Reads saved JSON/CSV artifacts from Google Drive and makes comparison tables/plots. This notebook does not train models and does not load checkpoints into GPU memory.

In [ ]:
import os, sys, platform
print('Python:', sys.version)
print('Platform:', platform.platform())

In [ ]:
!pip -q install pandas matplotlib seaborn

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, os
import pandas as pd

DRIVE_ROOT = Path('/content/drive/MyDrive/neurovlm_difumo_gat')
RUN_ROOT = DRIVE_ROOT / 'runs'
COMPARISON_FILE = RUN_ROOT / 'difumo_gat_comparison.csv'
OUT_FILE = RUN_ROOT / 'difumo_saved_artifact_summary.csv'
print('Run root:', RUN_ROOT)
print('Aggregate comparison CSV:', COMPARISON_FILE)

In [ ]:
def load_json(path):
    try:
        return json.loads(Path(path).read_text())
    except Exception:
        return {}

rows = []
for eval_path in sorted(RUN_ROOT.glob('*/*/eval_results.json')):
    run_dir = eval_path.parent
    batch = run_dir.parent.name
    run_name = run_dir.name
    config = load_json(run_dir / 'config.json')
    graph = load_json(run_dir / 'graph_stats.json')
    metrics_payload = load_json(eval_path)
    test = metrics_payload.get('test', {})
    val = metrics_payload.get('val', {})
    manifest = load_json(run_dir / 'artifacts_manifest.json')
    rows.append({
        'batch': batch,
        'run': run_name,
        'model': config.get('model'),
        'coefficient_source': config.get('coeff_source'),
        'ale_kernel_fwhm_mm': config.get('ale_kernel_fwhm_mm') if config.get('coeff_source') == 'ale_coordinates' else None,
        'graph_type': config.get('graph_type') if config.get('model') == 'gat' else 'none',
        'conv': config.get('conv') if config.get('model') == 'gat' else '',
        'hidden': config.get('hidden'),
        'heads': config.get('heads'),
        'layers': config.get('layers'),
        'batch_size': config.get('batch_size'),
        'lr_gat': config.get('lr_gat'),
        'lr_proj': config.get('lr_proj'),
        'temperature': config.get('temperature'),
        'test_auc': test.get('mean_auc'),
        'test_r@1': test.get('mean_recall@1'),
        'test_r@5': test.get('mean_recall@5'),
        'test_r@10': test.get('mean_recall@10'),
        'test_r@50': test.get('mean_recall@50'),
        'test_mrr': test.get('mean_mrr'),
        'test_median_rank': test.get('mean_median_rank'),
        'test_random_r@10': test.get('mean_random_recall@10'),
        'val_auc': val.get('mean_auc'),
        'n_edges': graph.get('number_of_edges'),
        'avg_degree': graph.get('average_degree'),
        'connected_components': graph.get('connected_components'),
        'edge_weight_mean': graph.get('edge_weight_mean'),
        'edge_weight_std': graph.get('edge_weight_std'),
        'brain_parameters': config.get('brain_parameters'),
        'peak_vram': manifest.get('comparison_row_payload', {}).get('peak_vram'),
        'training_time_per_epoch': manifest.get('comparison_row_payload', {}).get('training_time_per_epoch'),
        'run_dir': str(run_dir),
        'best_checkpoint': manifest.get('best_checkpoint', str(run_dir / 'checkpoints' / 'best_difumo_gat.pt')),
        'config_path': str(run_dir / 'config.json'),
        'eval_results_path': str(eval_path),
    })

summary = pd.DataFrame(rows)
if len(summary):
    summary = summary.sort_values('test_auc', ascending=False, na_position='last').reset_index(drop=True)
display(summary)
summary.to_csv(OUT_FILE, index=False)
print('Saved:', OUT_FILE)

In [ ]:
import matplotlib.pyplot as plt

if len(summary):
    plot_df = summary.dropna(subset=['test_auc']).copy()
    labels = plot_df['run'] + ' | ' + plot_df['coefficient_source'].fillna('') + ' | ' + plot_df['graph_type'].fillna('')
    fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(plot_df))))
    ax.barh(labels[::-1], plot_df['test_auc'][::-1])
    ax.set_xlabel('test full recall-curve AUC')
    ax.set_title('DiFuMo Saved-Run Retrieval AUC')
    fig.tight_layout()
    display(fig)
    fig.savefig(RUN_ROOT / 'difumo_saved_artifact_auc.png', dpi=160)
    plt.close(fig)

In [ ]:
if len(summary):
    metric_cols = ['run', 'model', 'coefficient_source', 'graph_type', 'test_auc', 'test_r@1', 'test_r@5', 'test_r@10', 'test_r@50', 'test_mrr', 'test_median_rank', 'test_random_r@10']
    display(summary[metric_cols])

## Interpretation Checklist

- Raw GAT > raw MLP: graph structure helps on the original DiFuMo coefficients.
- ALE-smoothed MLP > raw MLP: ALE smoothing improves the coefficient representation.
- ALE-smoothed GAT > ALE-smoothed MLP: graph structure helps after smoothing.
- Real-HCP GAT > coactivation/spatial GAT: graph quality matters.
- Shuffled/no-edge GAT ≈ real-graph GAT: the graph is not adding much.